# 02 -- DDPM + cosine latent loss, JOINT fine-tune of the encoder

Same loss as notebook 01, but the Xception encoder's weights ALSO train, with their own parameter group (`lr_encoder = 1e-5`, vs `lr = 2.26e-4` for the DDPM). The encoder module stays in `.eval()` mode throughout (BatchNorm/Dropout deterministic) even though its weights are being updated -- this avoids the running BN statistics drifting under the non-i.i.d. stream of denoised images the training loop feeds it.

Two mandatory guardrails are logged every epoch and plotted at the end:
1. **Latent collapse** -- mean per-dimension std of `z` over a probe batch. If this drops below 50% of its epoch-0 value, every cosine similarity trends to ~1 and the loss term becomes vacuous; a warning is printed when that happens.
2. **Inverse-task regression** -- per-parameter R^2 of the FULL encoder+head, recomputed each epoch on the held-out inverse-model test split. This is the real cost of joint fine-tuning: if R^2 degrades, generative fidelity was bought by damaging the inverse model, and this notebook shows it rather than hiding it.

**Kaggle input datasets required (attach all four):**
- `carloscanamejoy/dataset-spines-united-v2` -> `dataset_unificado_v2.npz`
- `carloscanamejoy/weights-xception-model` -> `xception_regressor_torch.pt`
- `carloscanamejoy/weights-models` -> `ddpm_spines_final_39crop.pt`
- `carloscanamejoy/physicalmetrics` -> `metrics.py`

Outputs are written to `/kaggle/working/ddpm_cosine_joint_finetune/`.

## 1. Environment setup

In [ ]:
# Check GPU availability
import os
from pathlib import Path

if Path('/kaggle').exists():
    print('Running on Kaggle')
else:
    print('WARNING: this does not look like Kaggle; continuing anyway')

try:
    import subprocess
    subprocess.run(['nvidia-smi'], check=False)
except Exception as e:
    print(f'Could not run nvidia-smi: {e}')


In [ ]:
# Dependencies. Kaggle usually ships torch/sklearn/matplotlib.
# pytorch-msssim may be missing; install it with a skimage SSIM fallback if the
# install fails (e.g. no internet on the Kaggle session).
try:
    import pytorch_msssim  # noqa: F401
    print('pytorch-msssim available')
except Exception:
    print('pytorch-msssim not installed; attempting install...')
    try:
        import sys, subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
        print('pytorch-msssim installed')
    except Exception as e:
        print(f'WARNING: could not install pytorch-msssim ({e}). Falling back to skimage SSIM.')

In [ ]:
# On Kaggle, datasets are mounted under /kaggle/input -- no kaggle.json needed.
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
if not KAGGLE_INPUT.exists():
    raise RuntimeError('/kaggle/input does not exist. Attach the datasets from "Add Input" on Kaggle.')

print('Mounted datasets:')
for d in sorted([x for x in KAGGLE_INPUT.iterdir() if x.is_dir()]):
    print(' -', d.name)


In [ ]:
# Resolve every required input file by name -- never hardcode a Kaggle path,
# dataset version suffixes change the mount directory name.
import glob


def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(
            f'{name} not found under /kaggle/input. '
            f'Attach the dataset that ships it (see the header markdown cell).'
        )
    return hits[0]


DATASET_PATH  = find_file('dataset_unificado_v2.npz')
METRICS_PATH  = find_file('metrics.py')
ENCODER_CKPT  = find_file('xception_regressor_torch.pt')
DDPM_CKPT     = find_file('ddpm_spines_final_39crop.pt')

print(f'DATASET_PATH : {DATASET_PATH}')
print(f'METRICS_PATH : {METRICS_PATH}')
print(f'ENCODER_CKPT : {ENCODER_CKPT}')
print(f'DDPM_CKPT    : {DDPM_CKPT}')


## 2. Imports and global configuration

In [ ]:
import os
import gc
import sys
import json
import time
import math
import pickle
import random
import importlib.util
import warnings

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

try:
    from pytorch_msssim import ssim as ssim_fn
    SSIM_BACKEND = 'pytorch_msssim'
except Exception:
    from skimage.metrics import structural_similarity as skimage_ssim
    SSIM_BACKEND = 'skimage_fallback'

    def ssim_fn(x, y, data_range=1.0, size_average=True):
        x_np = x.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()
        vals = [skimage_ssim(a[0], b[0], data_range=data_range) for a, b in zip(x_np, y_np)]
        val = float(np.mean(vals)) if size_average else np.asarray(vals, dtype=np.float32)
        return torch.as_tensor(val)

warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device : {DEVICE}')
print(f'SSIM   : {SSIM_BACKEND}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


In [ ]:
# Load metrics.py by path and register it as sys.modules['metrics'] -- the same
# idiom used by the rest of the repo's notebooks.
spec = importlib.util.spec_from_file_location('metrics', METRICS_PATH)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules['metrics'] = metrics

# Only the canonical three physical metrics exist in metrics.py; the retired
# susceptibility/specific-heat/exchange-energy metrics were removed upstream and are
# never imported here.
from metrics import (
    MASK, IMG_SIZE as PHYS_SIZE, DDPM_SIZE as PHYS_CANVAS_SIZE,
    topleft_crop, magnetization, spin_correlation, peak_wave_vector,
    physical_metrics_batch, PHYSICAL_METRIC_NAMES, PHYSICAL_METRIC_LABELS,
    masked_mse, masked_ssim, get_structure_label, apply_figure_style, save_figure,
    PARAM_NAMES, PARAM_INDEX,
)

apply_figure_style()
print(f'metrics module loaded from {METRICS_PATH}')
print(f'MASK: {MASK.shape}, disk pixels = {int(MASK.sum())}')
print(f'Physical metrics (canonical set of three): {PHYSICAL_METRIC_NAMES}')


In [ ]:
# DDPM hyperparameters -- identical to the published checkpoint's training run.
BEST_HPARAMS = {
    'lr':            2.2640535194211016e-04,
    'batch_size':    128,
    'base_ch':       80,
    'cond_emb_dim':  128,
    'dropout':       0.1,
    'beta_schedule': 'cosine',
    'ema_decay':     0.999,
    'weight_decay':  4.279388675327132e-05,
    'min_snr_gamma': 5.0,
}

WARMUP_EPOCHS  = 3      # linear LR warmup over the first epochs of this fine-tune
GRAD_CLIP_NORM = 0.5    # gradient-norm clip, matches the base DDPM training run

IMG_SIZE  = 40   # DDPM canvas (39x39 physical image reflect-padded to 40x40)
CROP_TO   = 39   # physical image size -- ALL physical metrics use this size
COND_DIM  = 8
T_STEPS    = 1000
BETA_START = 1e-4
BETA_END   = 0.02

# --- Fine-tune schedule --------------------------------------------------
# This notebook WARM-STARTS from the published DDPM checkpoint (DDPM_CKPT) and
# continues training with the cosine-similarity term added to the loss -- it does
# not retrain the diffusion model from scratch.
FINETUNE_EPOCHS   = 15
VAL_BATCHES       = 8      # batches per epoch on which fast_sample is run for val SSIM/MAE/MSE
VAL_FAST_STEPS     = 100

# --- Cosine (latent) loss -------------------------------------------------
# L = L_denoise + lambda_cos * (1 - cos(z(x0_pred_39), z(x0_39)))
# No physical term anywhere in the loss (see repo SPEC section 1).
COSINE_T_MAX = 400
# Only t < COSINE_T_MAX contributes to the cosine term. At high t, x0_pred is a
# clamped, near-pure-noise reconstruction and its latent carries no usable signal
# -- this is the known failure mode the t-window guards against (SPEC 1.1).

LAMBDA_COS_FINAL = 0.1
# Final weight of the cosine term once the ramp is complete. Raising this pushes
# the DDPM harder towards generating images whose encoder latent matches the
# conditioning image's latent, at the cost of denoising fidelity if pushed too far.
LAMBDA_WARMUP_EPOCHS = 5
# lambda_cos ramps 0 -> LAMBDA_COS_FINAL with a cosine ramp over this many epochs,
# so the DDPM has time to reach a reasonable denoising baseline before the cosine
# term starts pulling on it.

COSINE_EVERY_N_STEPS = 1
# Apply the cosine term (and the encoder forward pass it requires) only on every
# Nth optimizer step. The encoder forward is the dominant per-step cost, so raising
# this trades fidelity of the cosine signal for wall-clock speed -- e.g. N=4 cuts
# ~75% of encoder forward passes at the cost of a noisier gradient estimate.

TEST_FRACTION = 1.0
# Fraction of the internal test split used by the final evaluation block. Lower
# this (e.g. 0.05) for a cheap smoke run of the notebook end-to-end.

def lambda_cos_schedule(epoch):
    """Cosine ramp 0 -> LAMBDA_COS_FINAL over LAMBDA_WARMUP_EPOCHS epochs (epoch is 1-indexed)."""
    u = min(max(float(epoch) / max(LAMBDA_WARMUP_EPOCHS, 1), 0.0), 1.0)
    ramp = 0.5 - 0.5 * math.cos(math.pi * u)
    return LAMBDA_COS_FINAL * ramp

# Output
WORK_DIR = '/kaggle/working/ddpm_cosine_joint_finetune'
os.makedirs(WORK_DIR, exist_ok=True)
CKPT_OUT = f'{WORK_DIR}/ddpm_cosine_joint_finetune.pt'
HIST_OUT = f'{WORK_DIR}/history.json'
METRICS_OUT = f'{WORK_DIR}/final_metrics.json'

print('Config loaded:')
for k, v in BEST_HPARAMS.items():
    print(f'  {k:16s} = {v}')
print(f'  FINETUNE_EPOCHS   = {FINETUNE_EPOCHS}')
print(f'  COSINE_T_MAX      = {COSINE_T_MAX}')
print(f'  LAMBDA_COS_FINAL  = {LAMBDA_COS_FINAL}  over {LAMBDA_WARMUP_EPOCHS} warmup epochs')
print(f'  COSINE_EVERY_N_STEPS = {COSINE_EVERY_N_STEPS}')
print(f'  WORK_DIR          = {WORK_DIR}')


## 3. Dataset: load, global normalisation constants, 70/15/15 split

In [ ]:
data   = np.load(DATASET_PATH)
imgs   = data['img'].astype(np.float32)
params = data['params'].astype(np.float32)
labels = np.asarray(data['labels']) if 'labels' in data.files else None
if imgs.ndim == 3:
    imgs = imgs[..., np.newaxis]

N = len(imgs)
print(f'Dataset total: {N:,}')
print(f'  imgs   : {imgs.shape}  dtype={imgs.dtype}  range=[{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'  params : {params.shape}  dtype={params.dtype}')
if labels is not None:
    print(f'  labels : {labels.shape}  unique clusters = {sorted(set(labels.tolist()))}')
else:
    print('  labels : not present in this .npz -- per-phase breakdown will be skipped')

# Global min/max of the RAW physical images -- these are the constants the DDPM
# dataset used for its [-1, 1] normalisation (SPEC 0.4), and are needed again to
# undo that normalisation before feeding a DDPM output to the Xception encoder.
IMG_MIN = float(imgs.min())
IMG_MAX = float(imgs.max())
print(f'  IMG_MIN={IMG_MIN:.4f}  IMG_MAX={IMG_MAX:.4f}')


In [ ]:
def make_split(subsample_frac, seed=SEED):
    """70/15/15 split with a MinMaxScaler fitted on the train fold (DDPM conditioning scaler)."""
    rng = np.random.RandomState(seed)
    sub_idx = rng.choice(N, size=int(N * subsample_frac), replace=False)
    imgs_s, params_s = imgs[sub_idx], params[sub_idx]

    idx_all = np.arange(len(sub_idx))
    idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.30, random_state=seed)
    idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=seed)

    sc = MinMaxScaler()
    p_tr = sc.fit_transform(params_s[idx_tr]).astype(np.float32)
    p_va = sc.transform(params_s[idx_va]).astype(np.float32)
    p_te = sc.transform(params_s[idx_te]).astype(np.float32)

    return {
        'imgs_tr': imgs_s[idx_tr], 'p_tr': p_tr, 'idx_tr': sub_idx[idx_tr],
        'imgs_va': imgs_s[idx_va], 'p_va': p_va, 'idx_va': sub_idx[idx_va],
        'imgs_te': imgs_s[idx_te], 'p_te': p_te, 'idx_te': sub_idx[idx_te],
        'scaler': sc,
    }


SPLIT = make_split(subsample_frac=1.0, seed=SEED)
print(f"train={len(SPLIT['p_tr']):,}  val={len(SPLIT['p_va']):,}  test={len(SPLIT['p_te']):,}")


## 4. PyTorch dataset (reflect-pad 39x39 -> 40x40, no interpolation)

In [ ]:
class SpinesDataset(Dataset):
    """39x39 image reflect-padded to 40x40: F.pad(x, (0, 1, 0, 1), mode='reflect').
    Padding is applied on the RIGHT and BOTTOM edges only, so
    ``img[..., :39, :39]`` is an exact top-left crop back to the original pixels
    (never a centre crop -- see metrics.topleft_crop).
    """

    def __init__(self, imgs_arr, params_arr, img_size=40):
        imgs_t = torch.from_numpy(imgs_arr).permute(0, 3, 1, 2).float()
        H, W = imgs_t.shape[-2], imgs_t.shape[-1]
        if H != img_size or W != img_size:
            pad_h, pad_w = img_size - H, img_size - W
            assert pad_h >= 0 and pad_w >= 0
            imgs_t = F.pad(imgs_t, (0, pad_w, 0, pad_h), mode='reflect')
        mn, mx = imgs_t.min(), imgs_t.max()
        imgs_t = (imgs_t - mn) / (mx - mn + 1e-8)
        imgs_t = imgs_t * 2.0 - 1.0
        self.imgs = imgs_t.float()
        self.params = torch.from_numpy(params_arr).float()

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        return self.imgs[i], self.params[i]


def make_dataloaders(split_dict, batch_size, num_workers=2):
    ds_tr = SpinesDataset(split_dict['imgs_tr'], split_dict['p_tr'], IMG_SIZE)
    ds_va = SpinesDataset(split_dict['imgs_va'], split_dict['p_va'], IMG_SIZE)
    ds_te = SpinesDataset(split_dict['imgs_te'], split_dict['p_te'], IMG_SIZE)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,
                        num_workers=num_workers, pin_memory=True, drop_last=True)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    dl_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    return ds_tr, ds_va, ds_te, dl_tr, dl_va, dl_te


_ds_tr, _, _, _dl_tr, _, _ = make_dataloaders(SPLIT, batch_size=BEST_HPARAMS['batch_size'])
_x, _y = next(iter(_dl_tr))
print(f'Smoke test -- img: {_x.shape} [{_x.min():.2f}, {_x.max():.2f}]  cond: {_y.shape}')
del _ds_tr, _dl_tr, _x, _y


## 5. Noise schedule (cosine, per BEST_HPARAMS)

In [ ]:
class DDPMScheduler:
    """Beta schedule for DDPM. Supports 'linear' and 'cosine'."""

    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule='linear', device='cpu'):
        self.T = T
        self.schedule = schedule
        if schedule == 'linear':
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == 'cosine':
            # Nichol & Dhariwal 2021
            steps = T + 1
            s = 0.008
            x = torch.linspace(0, T, steps, device=device)
            alphas_cumprod = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            betas = betas.clamp(max=0.999)
        else:
            raise ValueError(f'Unknown schedule: {schedule}')

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.sqrt_alphas_cumprod = alphas_cumprod.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - alphas_cumprod).sqrt()
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.sqrt_recip_alphas = (1.0 / alphas).sqrt()
        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.snr = alphas_cumprod / (1.0 - alphas_cumprod)  # SNR_t, for min-SNR weighting

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_a = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_1a = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_a * x0 + sqrt_1a * noise, noise

    def min_snr_weight(self, t, gamma=5.0):
        snr_t = self.snr[t]
        return torch.clamp(snr_t, max=gamma) / snr_t

    @torch.no_grad()
    def p_sample_step(self, model, x_t, t_scalar, cond):
        B = x_t.shape[0]
        t_tensor = torch.full((B,), t_scalar, device=x_t.device, dtype=torch.long)
        eps_pred = model(x_t, t_tensor, cond)
        beta_t = self.betas[t_scalar]
        sqrt_ra = self.sqrt_recip_alphas[t_scalar]
        sqrt_1ma = self.sqrt_one_minus_alphas_cumprod[t_scalar]
        mean = sqrt_ra * (x_t - beta_t / sqrt_1ma * eps_pred)
        if t_scalar > 0:
            z = torch.randn_like(x_t)
            sigma = self.posterior_variance[t_scalar].sqrt()
            return mean + sigma * z
        return mean


_sch = DDPMScheduler(T=T_STEPS, schedule=BEST_HPARAMS['beta_schedule'], device=DEVICE)
print(f"Scheduler '{BEST_HPARAMS['beta_schedule']}' OK, "
      f'alphas_cumprod[0]={_sch.alphas_cumprod[0]:.6f}  alphas_cumprod[T-1]={_sch.alphas_cumprod[-1]:.6f}')
del _sch


## 6. Conditional U-Net

In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1))
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)


class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))

    def forward(self, t, cond):
        t_emb = sinusoidal_embedding(t, self.t_mlp[0].in_features)
        return self.t_mlp(t_emb) + self.c_mlp(cond)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        q, k, v = q.reshape(B, C, -1), k.reshape(B, C, -1), v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) / math.sqrt(C), dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(B, C, H, W)
        return x + self.proj(out)


class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1, 2, 4), cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        t_dim = emb_dim
        chs = [base_ch * m for m in ch_mults]
        self.emb = TimeCondEmbedding(t_dim=t_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)

        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        in_ch = chs[0]
        self.skip_channels = []
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout),
            ]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(
                nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs) - 1 else nn.Identity()
            )
            in_ch = out_ch

        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)

        self.up_blocks = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i + 1)]
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch + skip_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout),
            ]))
            self.up_samples.append(
                nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs) - 1 else nn.Identity()
            )
            in_ch = out_ch

        self.norm_out = nn.GroupNorm(8, chs[0])
        self.conv_out = nn.Conv2d(chs[0], img_channels, 1)

    def forward(self, x, t, cond):
        emb = self.emb(t, cond)
        h = self.conv_in(x)
        skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb)
            skips.append(h)
            h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, skip in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, skip], dim=1)
            h = rb1(h, emb); h = rb2(h, emb)
            h = us(h)
        h = F.silu(self.norm_out(h))
        return self.conv_out(h)


_m = ConditionalUNet(base_ch=BEST_HPARAMS['base_ch'], emb_dim=BEST_HPARAMS['cond_emb_dim'], dropout=BEST_HPARAMS['dropout']).to(DEVICE)
_n = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'U-Net: {_n/1e6:.2f}M params')
del _m
torch.cuda.empty_cache()


## 7. EMA and sampling helpers

In [ ]:
class EMA:
    """Simple EMA over model parameters."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    @torch.no_grad()
    def store_and_copy_to(self, model):
        self._backup = {n: p.data.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                p.data.copy_(self.shadow[n])

    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self._backup:
                p.data.copy_(self._backup[n])
        self._backup = {}


@torch.no_grad()
def compute_image_metrics(x_gen, x_real):
    """Plain image-space metrics on the 40x40 canvas (MAE/MSE/SSIM) -- NOT physical metrics."""
    mae = (x_gen - x_real).abs().mean().item()
    mse = ((x_gen - x_real) ** 2).mean().item()
    x_g01 = (x_gen + 1.0) / 2.0
    x_r01 = (x_real + 1.0) / 2.0
    ssim_val = ssim_fn(x_g01, x_r01, data_range=1.0, size_average=True).item()
    return mae, mse, ssim_val


@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=40):
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_tensor = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        eps_pred = model(x, t_tensor, cond)
        sqrt_a = scheduler.sqrt_alphas_cumprod[t_val]
        sqrt_1a = scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0_pred = ((x - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            sqrt_a_prev = scheduler.sqrt_alphas_cumprod[t_prev]
            sqrt_1a_prev = scheduler.sqrt_one_minus_alphas_cumprod[t_prev]
            x = sqrt_a_prev * x0_pred + sqrt_1a_prev * eps_pred
        else:
            x = x0_pred
    return x


print('DDPM core (scheduler, U-Net, EMA, fast_sample) defined.')


## 8. Load the published DDPM checkpoint

In [ ]:
model = ConditionalUNet(
    img_channels=1,
    base_ch=BEST_HPARAMS['base_ch'],
    ch_mults=(1, 2, 4),
    cond_dim=COND_DIM,
    emb_dim=BEST_HPARAMS['cond_emb_dim'],
    dropout=BEST_HPARAMS['dropout'],
).to(DEVICE)

scheduler = DDPMScheduler(T=T_STEPS, beta_start=BETA_START, beta_end=BETA_END,
                           schedule=BEST_HPARAMS['beta_schedule'], device=DEVICE)

_ckpt = torch.load(DDPM_CKPT, map_location=DEVICE, weights_only=False)
_state = _ckpt['model'] if isinstance(_ckpt, dict) and 'model' in _ckpt else _ckpt
# Strict by design. This notebook WARM-STARTS from the published checkpoint: a silent
# partial load would leave the U-Net partly randomly initialised while every log line
# still claimed a warm start. A key or shape mismatch means the checkpoint was not
# written by this architecture, and that must stop the run, not print a counter.
model.load_state_dict(_state, strict=True)
print(f'Loaded DDPM checkpoint (strict=True): {DDPM_CKPT}')
print(f'  {len(_state)} tensors into ConditionalUNet '
      f'({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)')

ema = EMA(model, decay=BEST_HPARAMS['ema_decay'])
if isinstance(_ckpt, dict) and _ckpt.get('ema') is not None:
    _ema_skipped = [k for k in _ckpt['ema'] if k not in ema.shadow]
    if _ema_skipped:
        raise RuntimeError(
            f'EMA shadow keys not present in the model: {_ema_skipped[:8]} '
            f'({len(_ema_skipped)} total). The checkpoint does not match this architecture.')
    for k, v in _ckpt['ema'].items():
        ema.shadow[k].copy_(v.to(DEVICE))
    print('  EMA shadow weights loaded from checkpoint.')
else:
    print('  WARNING: no EMA state in checkpoint; EMA shadow initialised from loaded weights.')


## 9. Xception inverse-model encoder

`xception_regressor_torch.pt` is **not** a generic checkpoint: it is produced by
`notebooks/inverse/xception-keras-to-torch.ipynb`, which ports
`modelo_xception_fulldatabaseV3100.h5` layer by layer into the module defined below and only
saves it once the Keras/PyTorch parity test passes (max abs difference < 1e-3 on real images).

So the builder has to be that exact module. Its state-dict keys are flat
(`conv1`, `bn1`, `b2_sc1`, `mid_sc.*`, `head_fc1`, ...) and they do **not** match a timm
`legacy_xception` backbone, where the weights live under `backbone.*` and `bn1`/`bn2` are the
entry-flow `BatchNorm2d(32)`/`BatchNorm2d(64)` rather than the head's `BatchNorm1d(2048)`/
`BatchNorm1d(256)`. Loading this checkpoint into a timm-based module fails with

```
size mismatch for bn1.weight: copying a param with shape torch.Size([32]) from checkpoint,
the shape in current model is torch.Size([2048]).
```

and with `strict=False` it would be worse than a crash: the name collision is the *only* thing
that errors, every backbone weight would be silently dropped and the encoder would run at random
initialisation. Hence `strict=True` below — a mismatch must be loud.

Two details of the port are load-bearing and are reproduced here:

- **TF `'same'` max-pool padding.** Keras splits the padding asymmetrically; a symmetric
  `MaxPool2d(padding=1)` shifts the feature map. `tf_same_maxpool` pads with `-inf` so the
  artificial cells never win a maximum.
- **BatchNorm epsilon.** Keras defaults to `1e-3`, PyTorch to `1e-5`. The conversion notebook
  copied Keras' value onto each module, but `eps` is a plain Python attribute and is **not**
  stored in the state dict — rebuilding the module here has to set it again, otherwise the
  ported weights quietly drift from the Keras evaluator.

In [ ]:
# --- Xception inverse model: exact PyTorch port of the Keras regressor ------
# Mirrors notebooks/inverse/xception-keras-to-torch.ipynb, which is what wrote
# ENCODER_CKPT. Any change here must be mirrored there or the state dict stops loading.

# Keras BatchNormalization defaults to epsilon=1e-3; PyTorch defaults to 1e-5.
# `eps` is not part of the state dict, so it has to be re-declared at build time.
KERAS_BN_EPS = 1e-3


def _bn2d(c):
    return nn.BatchNorm2d(c, eps=KERAS_BN_EPS)


def _bn1d(c):
    return nn.BatchNorm1d(c, eps=KERAS_BN_EPS)


def tf_same_maxpool(x, k=3, s=2):
    """MaxPooling2D(k, strides=s, padding='same') as TensorFlow computes it.

    TF distributes 'same' padding asymmetrically when the input size is even, which
    PyTorch's symmetric `padding=` cannot express. Padding with -inf reproduces how TF
    ignores the artificial cells inside a maximum.
    """
    ih, iw = x.shape[-2], x.shape[-1]
    oh, ow = math.ceil(ih / s), math.ceil(iw / s)
    ph = max((oh - 1) * s + k - ih, 0)
    pw = max((ow - 1) * s + k - iw, 0)
    if ph or pw:
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2), value=float('-inf'))
    return F.max_pool2d(x, k, s)


class SepConv(nn.Module):
    """SeparableConv2D(out, 3, padding='same', use_bias=False): depthwise then pointwise."""

    def __init__(self, cin, cout):
        super().__init__()
        self.depthwise = nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False)
        self.pointwise = nn.Conv2d(cin, cout, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class XceptionRegressor(nn.Module):
    """Xception (Chollet 2017) + the 8-output regression head of the inverse model.

    Head graph (SPEC 0.4): GlobalAveragePooling2D -> BatchNorm -> Dropout(0.4)
    -> Dense(256, relu) -> BatchNorm -> Dropout(0.3) -> Dense(8, linear).
    ``forward_features`` returns the 256-d latent AFTER the ReLU of Dense(256) and
    BEFORE the final Dense(8) -- this is ``z`` used everywhere in the cosine loss.
    """

    def __init__(self, num_targets=8):
        super().__init__()
        # --- entry flow ---
        self.conv1 = nn.Conv2d(3, 32, 3, stride=2, bias=False)
        self.bn1 = _bn2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, bias=False)
        self.bn2 = _bn2d(64)

        self.res1 = nn.Conv2d(64, 128, 1, stride=2, bias=False)
        self.res1_bn = _bn2d(128)
        self.b2_sc1 = SepConv(64, 128)
        self.b2_bn1 = _bn2d(128)
        self.b2_sc2 = SepConv(128, 128)
        self.b2_bn2 = _bn2d(128)

        self.res2 = nn.Conv2d(128, 256, 1, stride=2, bias=False)
        self.res2_bn = _bn2d(256)
        self.b3_sc1 = SepConv(128, 256)
        self.b3_bn1 = _bn2d(256)
        self.b3_sc2 = SepConv(256, 256)
        self.b3_bn2 = _bn2d(256)

        self.res3 = nn.Conv2d(256, 728, 1, stride=2, bias=False)
        self.res3_bn = _bn2d(728)
        self.b4_sc1 = SepConv(256, 728)
        self.b4_bn1 = _bn2d(728)
        self.b4_sc2 = SepConv(728, 728)
        self.b4_bn2 = _bn2d(728)

        # --- middle flow: 8 identical blocks ---
        self.mid_sc = nn.ModuleList()
        self.mid_bn = nn.ModuleList()
        for _ in range(8):
            self.mid_sc.append(nn.ModuleList([SepConv(728, 728) for _ in range(3)]))
            self.mid_bn.append(nn.ModuleList([_bn2d(728) for _ in range(3)]))

        # --- exit flow ---
        self.res4 = nn.Conv2d(728, 1024, 1, stride=2, bias=False)
        self.res4_bn = _bn2d(1024)
        self.b13_sc1 = SepConv(728, 728)
        self.b13_bn1 = _bn2d(728)
        self.b13_sc2 = SepConv(728, 1024)
        self.b13_bn2 = _bn2d(1024)
        self.b14_sc1 = SepConv(1024, 1536)
        self.b14_bn1 = _bn2d(1536)
        self.b14_sc2 = SepConv(1536, 2048)
        self.b14_bn2 = _bn2d(2048)

        # --- regression head ---
        self.head_bn1 = _bn1d(2048)
        self.head_drop1 = nn.Dropout(0.4)
        self.head_fc1 = nn.Linear(2048, 256)
        self.head_bn2 = _bn1d(256)
        self.head_drop2 = nn.Dropout(0.3)
        self.head_out = nn.Linear(256, num_targets)

    def features(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        # block2: no activation before the first separable conv
        res = self.res1_bn(self.res1(x))
        x = self.b2_bn1(self.b2_sc1(x))
        x = self.b2_bn2(self.b2_sc2(F.relu(x)))
        x = tf_same_maxpool(x) + res

        for res_c, res_b, sc1, bn1, sc2, bn2 in [
            (self.res2, self.res2_bn, self.b3_sc1, self.b3_bn1, self.b3_sc2, self.b3_bn2),
            (self.res3, self.res3_bn, self.b4_sc1, self.b4_bn1, self.b4_sc2, self.b4_bn2),
        ]:
            res = res_b(res_c(x))
            x = bn1(sc1(F.relu(x)))
            x = bn2(sc2(F.relu(x)))
            x = tf_same_maxpool(x) + res

        for scs, bns in zip(self.mid_sc, self.mid_bn):
            res = x
            for sc, bn in zip(scs, bns):
                x = bn(sc(F.relu(x)))
            x = x + res

        res = self.res4_bn(self.res4(x))
        x = self.b13_bn1(self.b13_sc1(F.relu(x)))
        x = self.b13_bn2(self.b13_sc2(F.relu(x)))
        x = tf_same_maxpool(x) + res

        x = F.relu(self.b14_bn1(self.b14_sc1(x)))
        x = F.relu(self.b14_bn2(self.b14_sc2(x)))
        return x

    def forward_features(self, x):
        """Returns z, the 256-d latent (post-ReLU of Dense(256), pre-final-Dense)."""
        f = self.features(x).mean(dim=(2, 3))   # GlobalAveragePooling2D
        f = self.head_drop1(self.head_bn1(f))
        return F.relu(self.head_fc1(f))

    def forward(self, x):
        z = self.forward_features(x)
        h = self.head_drop2(self.head_bn2(z))
        return self.head_out(h)


def load_encoder_checkpoint(path, num_targets=8, device=DEVICE):
    """Load the ported Xception regressor.

    Strict by design. A key or shape mismatch means the checkpoint was not produced by
    the port above, and a silently half-initialised encoder would poison every latent
    downstream without ever raising.
    """
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    print(f'Checkpoint top-level type: {type(ckpt)}')

    if isinstance(ckpt, nn.Module):
        print('Checkpoint is a pickled nn.Module; using it directly.')
        return ckpt.to(device)

    if not isinstance(ckpt, dict):
        raise TypeError(f'Unrecognised checkpoint type: {type(ckpt)}')

    state_dict = None
    for key in ('state_dict', 'model', 'model_state_dict'):
        if key in ckpt and isinstance(ckpt[key], dict):
            state_dict = ckpt[key]
            print(f"Found state dict under key '{key}'.")
            break
    if state_dict is None:
        state_dict = ckpt  # bare OrderedDict / state_dict
        print('Treating the top-level dict itself as a bare state_dict.')

    n_out = int(ckpt.get('n_out', num_targets))
    if n_out != num_targets:
        raise ValueError(
            f'Checkpoint was converted with n_out={n_out} but the notebook asks for '
            f'num_targets={num_targets}.'
        )

    enc = XceptionRegressor(num_targets=num_targets)
    enc.load_state_dict(state_dict, strict=True)
    enc = enc.to(device)

    # Provenance written by the conversion notebook -- worth seeing before training on it.
    if 'parity_max_abs_diff' in ckpt:
        print(f"  Keras/Torch parity: max abs diff = {ckpt['parity_max_abs_diff']:.3e} "
              f"(ok={ckpt.get('parity_ok')})")
    if ckpt.get('parity_ok') is False:
        raise RuntimeError(
            'Checkpoint reports parity_ok=False against the Keras model. Re-run '
            'xception-keras-to-torch.ipynb before using it as the cycle encoder.'
        )
    if 'input_size' in ckpt and int(ckpt['input_size']) != 224:
        raise ValueError(
            f"Checkpoint expects input_size={ckpt['input_size']}, but phys_to_encoder_input "
            f'resizes to 224.'
        )
    if 'scalers_match' in ckpt:
        print(f"  DDPM/Xception scalers equivalent: {ckpt['scalers_match']}")

    print(f'  Loaded {len(state_dict)} tensors into XceptionRegressor (strict=True).')
    return enc


encoder = load_encoder_checkpoint(ENCODER_CKPT, num_targets=COND_DIM, device=DEVICE)
encoder.eval()  # BN/Dropout deterministic -- matters even when the weights train (see below)
print(f'Encoder loaded and set to .eval(). Trainable params: '
      f'{sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}')

### 9.1 Preprocessing -- undo the DDPM [-1, 1] normalisation, resize for Xception

```
x_phys = (x_ddpm39 + 1.0) / 2.0 * (mx - mn) + mn
x = F.interpolate(x_phys, size=(224, 224), mode='bilinear', align_corners=False)
x = x.repeat(1, 3, 1, 1)
```
`align_corners=False` matches `tf.image.resize`'s half-pixel convention (SPEC 0.4). No `keras.applications.xception.preprocess_input` and no extra rescaling -- the original training pipeline used the raw pixel range.

In [ ]:
def ddpm_norm_to_phys(x_norm, mn=IMG_MIN, mx=IMG_MAX):
    """Undo the DDPM's [-1, 1] normalisation back to the raw physical pixel range."""
    return (x_norm + 1.0) / 2.0 * (mx - mn) + mn


def phys_to_encoder_input(x_phys):
    """Raw physical pixels (B, 1, 39, 39) -> Xception input (B, 3, 224, 224)."""
    x = F.interpolate(x_phys, size=(224, 224), mode='bilinear', align_corners=False)
    return x.repeat(1, 3, 1, 1)


def encoder_preprocess(x_norm39):
    """DDPM-normalised (B, 1, 39, 39) in [-1, 1] -> Xception input (B, 3, 224, 224)."""
    return phys_to_encoder_input(ddpm_norm_to_phys(x_norm39))


def get_latent(enc, x_norm39):
    """z = the 256-d latent for a batch of DDPM-normalised, ALREADY-CROPPED-TO-39 images."""
    return enc.forward_features(encoder_preprocess(x_norm39))


print('Preprocessing + get_latent defined.')


### 9.2 Mandatory parity check

Published reference R² (from the original Keras training run): KDM 0.9498, J2 0.9146, T 0.8430. If the measured R² is far below these, the preprocessing or the checkpoint key mapping is wrong -- **the notebook stops rather than silently continuing.**

In [ ]:
# Replicate the inverse model's OWN train/val/test split (SPEC 0.4) -- this is a
# DIFFERENT split from the DDPM's 70/15/15 split, and is the split whose train fold
# the target MinMaxScaler was fit on.
_idx_all = np.arange(N)
_idx_trainval_inv, IDX_TEST_INV = train_test_split(_idx_all, test_size=0.15, random_state=42)
IDX_TRAIN_INV, IDX_VAL_INV = train_test_split(_idx_trainval_inv, test_size=0.1765, random_state=42)

INV_SCALER = MinMaxScaler().fit(params[IDX_TRAIN_INV])
print(f'Inverse-model split: train={len(IDX_TRAIN_INV):,}  val={len(IDX_VAL_INV):,}  test={len(IDX_TEST_INV):,}')


REFERENCE_R2 = {'KDM': 0.9498, 'J2': 0.9146, 'T': 0.8430}
R2_WARN_MARGIN = 0.15  # absolute R2 drop tolerated before the notebook halts loudly


@torch.no_grad()
def encoder_parity_r2(enc, idx_subset, batch_size=64, max_samples=1024):
    enc.eval()
    idx_subset = idx_subset[:max_samples]
    preds, targets = [], []
    for i in range(0, len(idx_subset), batch_size):
        idx_b = idx_subset[i:i + batch_size]
        x_phys = torch.from_numpy(imgs[idx_b]).permute(0, 3, 1, 2).float().to(DEVICE)
        y_true = INV_SCALER.transform(params[idx_b])
        x_in = phys_to_encoder_input(x_phys)
        y_pred = enc(x_in).cpu().numpy()
        preds.append(y_pred)
        targets.append(y_true)
    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)
    return r2_score(targets, preds, multioutput='raw_values'), preds, targets


_r2_per_param, _, _ = encoder_parity_r2(encoder, IDX_TEST_INV)
print('Per-parameter R^2 on the inverse-model held-out test split:')
for name, r2 in zip(PARAM_NAMES, _r2_per_param):
    print(f'  {name:8s} R^2 = {r2:.4f}')

# Loud, hard stop if parity is far below the published reference -- do not continue
# training or sampling on a mis-mapped encoder.
# Column indices come from metrics.PARAM_INDEX, the verified order of
# data['params']: T, Jex2, Jex3, Jex4, Kan1, KanS, Hex, KDM.
_param_idx = {'T': PARAM_INDEX['T'], 'J2': PARAM_INDEX['Jex2'], 'KDM': PARAM_INDEX['KDM']}
_bad = []
for ref_name, ref_val in REFERENCE_R2.items():
    if ref_name in _param_idx:
        measured = _r2_per_param[_param_idx[ref_name]]
        if measured < ref_val - R2_WARN_MARGIN:
            _bad.append((ref_name, ref_val, measured))
if _bad:
    msg = ' | '.join(f'{n}: expected~{ref:.4f} got {got:.4f}' for n, ref, got in _bad)
    raise RuntimeError(
        'ENCODER PARITY CHECK FAILED -- preprocessing or checkpoint key mapping is '
        f'almost certainly wrong. {msg}. STOPPING before any further training/sampling.'
    )
print('Parity check passed (within tolerance of published reference values).')


## 10. Loss -- no physical term

```
L = L_denoise + lambda_cos * (1 - cos(z(x0_pred_39), z(x0_39)))
```

- `L_denoise`: the existing min-SNR-weighted epsilon MSE, unchanged.
- `x0_pred = ((x_t - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)`, then top-left crop to 39.
- `z(.)`: the 256-d encoder latent. The target branch `z(x0_39)` is always detached.
- Cosine term is averaged over the batch with a **t-window mask**: only `t < COSINE_T_MAX` contributes, because at high `t`, `x0_pred` is a clamped, near-pure-noise reconstruction whose latent carries no usable signal (SPEC 1.1 -- the known failure mode).

In [ ]:
def ddpm_cosine_loss(unet, enc, x0, cond, scheduler, min_snr_gamma, lambda_cos,
                      step_idx=0, cosine_every_n=COSINE_EVERY_N_STEPS, return_parts=False):
    B = x0.shape[0]
    t = torch.randint(0, scheduler.T, (B,), device=x0.device)
    noise = torch.randn_like(x0)
    x_t, noise_added = scheduler.q_sample(x0, t, noise)
    eps_pred = unet(x_t, t, cond)

    per_sample_mse = ((eps_pred - noise_added) ** 2).mean(dim=[1, 2, 3])
    w_snr = scheduler.min_snr_weight(t, gamma=min_snr_gamma)
    denoise_loss = (w_snr * per_sample_mse).mean()

    cos_loss = x0.new_tensor(0.0)
    window_frac = 0.0
    run_cosine_step = (lambda_cos > 0) and (cosine_every_n <= 1 or step_idx % cosine_every_n == 0)

    if run_cosine_step:
        sqrt_a = scheduler.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_1a = scheduler.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        x0_pred = ((x_t - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)

        # Crop to 39 BEFORE anything touches the encoder -- the encoder (and every
        # physical metric downstream) only ever sees the 39x39 physical image.
        x0_pred_39 = topleft_crop(x0_pred)
        x0_39 = topleft_crop(x0)

        # Select the in-window samples BEFORE the encoder pass. The old form ran the
        # encoder over the whole batch and then multiplied by a 0/1 mask, discarding
        # ~60% of the dominant per-step cost at COSINE_T_MAX=400 of T=1000. Subsetting
        # is safe because the encoder is always in .eval(): its BatchNorm uses running
        # statistics, so batch composition cannot change any individual result.
        in_window = t < COSINE_T_MAX
        window_frac = in_window.float().mean().item()

        if in_window.any():
            sel = in_window.nonzero(as_tuple=True)[0]
            z_pred = get_latent(enc, x0_pred_39[sel])
            with torch.no_grad():
                z_true = get_latent(enc, x0_39[sel]).detach()
            cos_sim = F.cosine_similarity(z_pred, z_true, dim=1, eps=1e-8)
            # Mean over the selected samples is identical to the old
            # sum((1-cos)*w)/sum(w), which was already a masked mean.
            cos_loss = (1.0 - cos_sim).mean()

    total_loss = denoise_loss + lambda_cos * cos_loss

    if not return_parts:
        return total_loss
    parts = {
        'total': total_loss.detach(),
        'denoise': denoise_loss.detach(),
        'cosine': cos_loss.detach() if torch.is_tensor(cos_loss) else torch.as_tensor(cos_loss),
        'lambda_cos': torch.as_tensor(lambda_cos),
        'window_frac': torch.as_tensor(window_frac),
    }
    return total_loss, parts


print('ddpm_cosine_loss defined (t-window mask on COSINE_T_MAX, lambda ramp via lambda_cos_schedule).')


## 11. Encoder trainability -- JOINT (encoder weights train, BN/Dropout stay in eval mode)

The encoder is NOT frozen: its parameters keep `requires_grad=True` and get their own optimiser param group at `lr_encoder = 1e-5`. `encoder.eval()` is still called (and re-asserted every epoch in the training loop) so BatchNorm/Dropout remain deterministic while the underlying weights are updated by gradient steps.

In [ ]:
encoder.requires_grad_(True)
encoder.eval()  # BN/Dropout deterministic even though weights train

n_trainable_encoder = sum(p.requires_grad for p in encoder.parameters())
assert n_trainable_encoder > 0, 'Encoder should be trainable in the joint fine-tune notebook'
print(f'Encoder trainable: {n_trainable_encoder} trainable parameter tensors.')

MODEL_NAME = 'DDPM+cos (joint)'
LR_ENCODER = 1e-5  # much smaller than the DDPM LR -- the encoder is pretrained, not learned from scratch

optimizer = torch.optim.AdamW(
    [{'name': 'ddpm', 'params': model.parameters(), 'lr': BEST_HPARAMS['lr'], 'weight_decay': BEST_HPARAMS['weight_decay']}],
)
encoder_optimizer = torch.optim.AdamW(
    [{'name': 'encoder', 'params': encoder.parameters(), 'lr': LR_ENCODER, 'weight_decay': 1e-5}],
)
cosine_lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(FINETUNE_EPOCHS - WARMUP_EPOCHS, 1), eta_min=BEST_HPARAMS['lr'] * 0.1,
)

# The encoder gets the SAME linear warmup as the DDPM (scaled to its own target LR) and
# its OWN EMA, separate from the DDPM's EMA object.
encoder_ema = EMA(encoder, decay=BEST_HPARAMS['ema_decay'])
print(f'DDPM optimiser lr={BEST_HPARAMS["lr"]:.2e}   Encoder optimiser lr={LR_ENCODER:.2e}')
print('Encoder has its own EMA and shares the DDPM warmup schedule (scaled to LR_ENCODER).')


## 11.1 Guardrails: latent collapse + inverse-task R^2

In [ ]:
PROBE_IMGS = torch.from_numpy(SPLIT['imgs_va'][:64]).permute(0, 3, 1, 2).float().to(DEVICE)
PROBE_IMGS_NORM = (PROBE_IMGS - IMG_MIN) / (IMG_MAX - IMG_MIN + 1e-8) * 2.0 - 1.0


@torch.no_grad()
def latent_collapse_std(enc, probe=PROBE_IMGS_NORM):
    """Mean per-dimension std of z over the probe batch -- a collapsing encoder makes
    every cosine similarity trend to ~1 and the cosine loss term vacuous."""
    z = get_latent(enc, probe)
    return float(z.std(dim=0).mean().item())


EPOCH0_LATENT_STD = latent_collapse_std(encoder)
print(f'Epoch-0 latent std (baseline for the collapse guardrail): {EPOCH0_LATENT_STD:.4f}')


def guardrails_fn(enc, epoch):
    latent_std = latent_collapse_std(enc)
    if latent_std < 0.5 * EPOCH0_LATENT_STD:
        print(f'  WARNING: latent std {latent_std:.4f} < 50% of epoch-0 baseline '
              f'{EPOCH0_LATENT_STD:.4f} -- possible latent collapse.')
    r2_vals, _, _ = encoder_parity_r2(enc, IDX_TEST_INV, max_samples=256)
    inv_r2_mean = float(np.mean(r2_vals))
    return {'latent_std': latent_std, 'inv_r2_mean': inv_r2_mean}


print('Guardrails defined: latent_collapse_std, inverse-task R^2 via encoder_parity_r2.')


## 12. Training loop

In [ ]:
def run_training(unet, enc, sched, optim, lr_sched, split_dict, epochs,
                  val_batches, val_fast_steps, use_ema, encoder_optim=None, guardrails=None,
                  encoder_ema=None, encoder_lr=None):
    torch.manual_seed(SEED)
    _, _, _, dl_tr, dl_va, _ = make_dataloaders(split_dict, batch_size=BEST_HPARAMS['batch_size'])
    scaler_amp = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    min_snr_gamma = BEST_HPARAMS['min_snr_gamma']

    history = {'train_loss': [], 'train_denoise': [], 'train_cosine': [], 'window_frac': [],
               'val_loss': [], 'val_ssim': [], 'val_mae': [], 'val_mse': [], 'lambda_cos': [], 'lr': []}
    if guardrails is not None:
        history.update({'latent_std': [], 'inv_r2_mean': []})

    best_val_ssim = -1.0
    best_state = None
    best_encoder_state = None
    step_idx = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        if epoch <= WARMUP_EPOCHS:
            lr_now = BEST_HPARAMS['lr'] * (epoch / WARMUP_EPOCHS)
            for pg in optim.param_groups:
                pg['lr'] = lr_now
            # Encoder shares the SAME linear warmup, scaled to its own target LR (SPEC 2.2).
            if encoder_optim is not None and encoder_lr is not None:
                enc_lr_now = encoder_lr * (epoch / WARMUP_EPOCHS)
                for pg in encoder_optim.param_groups:
                    pg['lr'] = enc_lr_now

        lambda_cos = lambda_cos_schedule(epoch)
        unet.train()
        if encoder_optim is not None:
            enc.train(False)  # keep BN/Dropout deterministic even while weights train

        tr_loss, tr_denoise, tr_cos, tr_wf = [], [], [], []
        for x0, cond in dl_tr:
            x0, cond = x0.to(DEVICE, non_blocking=True), cond.to(DEVICE, non_blocking=True)
            optim.zero_grad(set_to_none=True)
            if encoder_optim is not None:
                encoder_optim.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                loss, parts = ddpm_cosine_loss(unet, enc, x0, cond, sched, min_snr_gamma,
                                                lambda_cos, step_idx=step_idx, return_parts=True)
            if not torch.isfinite(loss):
                step_idx += 1
                continue
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(unet.parameters(), GRAD_CLIP_NORM)
            if encoder_optim is not None:
                scaler_amp.unscale_(encoder_optim)
                torch.nn.utils.clip_grad_norm_(enc.parameters(), GRAD_CLIP_NORM)
            scaler_amp.step(optim)
            if encoder_optim is not None:
                scaler_amp.step(encoder_optim)
            scaler_amp.update()
            if use_ema is not None:
                use_ema.update(unet)
            if encoder_ema is not None:
                encoder_ema.update(enc)
            tr_loss.append(float(parts['total']))
            tr_denoise.append(float(parts['denoise']))
            tr_cos.append(float(parts['cosine']))
            tr_wf.append(float(parts['window_frac']))
            step_idx += 1

        if use_ema is not None:
            use_ema.store_and_copy_to(unet)
        if encoder_ema is not None:
            encoder_ema.store_and_copy_to(enc)
            enc.eval()  # re-assert BN/Dropout determinism after the EMA weight swap
        unet.eval()
        val_losses, maes, mses, ssims = [], [], [], []
        for i, (x0, cond) in enumerate(dl_va):
            x0, cond = x0.to(DEVICE), cond.to(DEVICE)
            with torch.no_grad(), torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                vl = ddpm_cosine_loss(unet, enc, x0, cond, sched, min_snr_gamma, lambda_cos, step_idx=step_idx)
            val_losses.append(float(vl) if torch.isfinite(vl) else float('nan'))
            if i < val_batches:
                x_gen = fast_sample(unet, cond, sched, n_steps=val_fast_steps, img_size=IMG_SIZE)
                m, s, ss = compute_image_metrics(x_gen, x0)
                maes.append(m); mses.append(s); ssims.append(ss)
        if use_ema is not None:
            use_ema.restore(unet)
        if encoder_ema is not None:
            encoder_ema.restore(enc)

        if epoch > WARMUP_EPOCHS:
            lr_sched.step()

        val_ssim = float(np.mean(ssims)) if ssims else -1.0
        history['train_loss'].append(float(np.mean(tr_loss)) if tr_loss else float('nan'))
        history['train_denoise'].append(float(np.mean(tr_denoise)) if tr_denoise else float('nan'))
        history['train_cosine'].append(float(np.mean(tr_cos)) if tr_cos else float('nan'))
        history['window_frac'].append(float(np.mean(tr_wf)) if tr_wf else float('nan'))
        history['val_loss'].append(float(np.nanmean(val_losses)) if val_losses else float('nan'))
        history['val_ssim'].append(val_ssim)
        history['val_mae'].append(float(np.mean(maes)) if maes else float('nan'))
        history['val_mse'].append(float(np.mean(mses)) if mses else float('nan'))
        history['lambda_cos'].append(lambda_cos)
        history['lr'].append(optim.param_groups[0]['lr'])

        if guardrails is not None:
            g = guardrails(enc, epoch)
            history['latent_std'].append(g['latent_std'])
            history['inv_r2_mean'].append(g['inv_r2_mean'])

        improved = val_ssim > best_val_ssim
        if improved:
            best_val_ssim = val_ssim
            best_state = {k: v.detach().cpu().clone() for k, v in unet.state_dict().items()}
            if encoder_optim is not None:
                best_encoder_state = {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()}

        star = ' *' if improved else ''
        extra = ''
        if guardrails is not None:
            extra = f" latent_std={g['latent_std']:.4f} inv_R2={g['inv_r2_mean']:.4f}"
        print(f"Ep[{epoch:2d}/{epochs}] loss={history['train_loss'][-1]:.4f} "
              f"denoise={history['train_denoise'][-1]:.4f} cos={history['train_cosine'][-1]:.4f} "
              f"window_frac={history['window_frac'][-1]:.2f} lambda_cos={lambda_cos:.4f} "
              f"val_SSIM={val_ssim:.4f}{star} val_MAE={history['val_mae'][-1]:.4f}"
              f"{extra} {time.time() - t0:.0f}s")

    return {'history': history, 'best_val_ssim': best_val_ssim, 'best_state': best_state,
            'best_encoder_state': best_encoder_state}


train_result = run_training(
    model, encoder, scheduler, optimizer, cosine_lr_scheduler, SPLIT,
    epochs=FINETUNE_EPOCHS, val_batches=VAL_BATCHES, val_fast_steps=VAL_FAST_STEPS,
    use_ema=ema, encoder_optim=encoder_optimizer, guardrails=guardrails_fn,
    encoder_ema=encoder_ema, encoder_lr=LR_ENCODER,
)
if train_result['best_state'] is not None:
    model.load_state_dict(train_result['best_state'])
if train_result['best_encoder_state'] is not None:
    encoder.load_state_dict(train_result['best_encoder_state'])
    encoder.eval()
print(f"Best val SSIM: {train_result['best_val_ssim']:.4f}")


## 13. Save checkpoint

In [ ]:
torch.save({
    'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
    'ema': {k: v.detach().cpu() for k, v in ema.shadow.items()},
    'encoder': {k: v.detach().cpu() for k, v in encoder.state_dict().items()},
    'encoder_ema': {k: v.detach().cpu() for k, v in encoder_ema.shadow.items()},
    'hyperparams': BEST_HPARAMS,
    'model_name': MODEL_NAME,
    'encoder_frozen': False,
    'lr_encoder': LR_ENCODER,
    'finetune_epochs': FINETUNE_EPOCHS,
    'lambda_cos_final': LAMBDA_COS_FINAL,
    'cosine_t_max': COSINE_T_MAX,
    'seed': SEED,
}, CKPT_OUT)
with open(HIST_OUT, 'w') as f:
    json.dump(train_result['history'], f, indent=2)
print(f'Saved {CKPT_OUT}')
print(f'Saved {HIST_OUT}')


## 14. Training curves (including guardrails)

In [ ]:
hist = train_result['history']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes[0, 0].plot(hist['train_loss'], label='train total')
axes[0, 0].plot(hist['val_loss'], label='val total')
axes[0, 0].set_title('Total loss'); axes[0, 0].legend()
axes[0, 1].plot(hist['train_denoise'], label='denoise')
axes[0, 1].plot(hist['train_cosine'], label='cosine')
axes[0, 1].set_title('Loss components'); axes[0, 1].legend()
axes[0, 2].plot(hist['val_ssim'], label='val SSIM', color='tab:green')
axes[0, 2].set_title('Validation SSIM'); axes[0, 2].legend()
axes[1, 0].plot(hist['lambda_cos'], label='lambda_cos', color='tab:purple')
axes[1, 0].plot(hist['window_frac'], label='t-window fraction', color='tab:orange')
axes[1, 0].set_title('Cosine schedule'); axes[1, 0].legend()
axes[1, 1].plot(hist['latent_std'], label='latent std', color='tab:red')
axes[1, 1].axhline(0.5 * EPOCH0_LATENT_STD, color='k', ls='--', lw=1, label='50% of epoch-0 baseline')
axes[1, 1].set_title('Guardrail: latent collapse'); axes[1, 1].legend()
axes[1, 2].plot(hist['inv_r2_mean'], label='mean inverse-task R^2', color='tab:brown')
axes[1, 2].set_title('Guardrail: inverse-task R^2 (encoder+head)'); axes[1, 2].legend()
fig.suptitle(f'{MODEL_NAME}: training curves + guardrails')
save_figure(fig, f'{WORK_DIR}/training_curves')
plt.show()


## Final evaluation (shared protocol, `metrics.py` only)

Model under evaluation: **DDPM+cos (joint fine-tune)**. Every physical metric below is computed strictly on the 39x39 crop -- `topleft_crop` runs before any call into `metrics.py`. `TEST_FRACTION={TEST_FRACTION}`

In [ ]:
@torch.no_grad()
def generate_test_batch(unet, sched, cond_batch, n_steps=250, use_ema=None):
    if use_ema is not None:
        use_ema.store_and_copy_to(unet)
    unet.eval()
    x_gen = fast_sample(unet, cond_batch, sched, n_steps=n_steps, img_size=IMG_SIZE)
    if use_ema is not None:
        use_ema.restore(unet)
    return x_gen


_, _, ds_te, _, _, dl_te = make_dataloaders(SPLIT, batch_size=BEST_HPARAMS['batch_size'])
n_test_total = len(ds_te)
n_test_eval = max(1, int(n_test_total * TEST_FRACTION))
print(f'Evaluating on {n_test_eval:,} / {n_test_total:,} test samples (TEST_FRACTION={TEST_FRACTION}).')

orig_39_all, gen_39_all, cond_all, idx_all_te = [], [], [], []
n_seen = 0
for x0, cond in dl_te:
    if n_seen >= n_test_eval:
        break
    x0, cond = x0.to(DEVICE), cond.to(DEVICE)
    x_gen = generate_test_batch(model, scheduler, cond, n_steps=VAL_FAST_STEPS, use_ema=ema)
    # Crop to 39 FIRST, then everything downstream is physical-metric-safe.
    orig_39_all.append(topleft_crop(x0).cpu().numpy()[:, 0])
    gen_39_all.append(topleft_crop(x_gen).cpu().numpy()[:, 0])
    cond_all.append(cond.cpu().numpy())
    n_seen += x0.shape[0]

orig_39 = np.concatenate(orig_39_all, axis=0)[:n_test_eval]
gen_39 = np.concatenate(gen_39_all, axis=0)[:n_test_eval]
cond_te = np.concatenate(cond_all, axis=0)[:n_test_eval]
assert orig_39.shape[-2:] == (39, 39) and gen_39.shape[-2:] == (39, 39)
print(f'orig_39: {orig_39.shape}  gen_39: {gen_39.shape}')


### Physical metrics: original vs. generated (R^2 + parity plot)

In [ ]:
phys_orig = physical_metrics_batch(orig_39)
phys_gen = physical_metrics_batch(gen_39)

fig, axes = plt.subplots(1, len(PHYSICAL_METRIC_NAMES), figsize=(5 * len(PHYSICAL_METRIC_NAMES), 4.2))
phys_r2 = {}
for ax, name in zip(axes, PHYSICAL_METRIC_NAMES):
    o, g = phys_orig[name], phys_gen[name]
    valid = np.isfinite(o) & np.isfinite(g)
    r2 = r2_score(o[valid], g[valid]) if valid.sum() > 1 else float('nan')
    phys_r2[name] = r2
    ax.scatter(o[valid], g[valid], s=6, alpha=0.4)
    lims = [min(o[valid].min(), g[valid].min()), max(o[valid].max(), g[valid].max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel(f'{PHYSICAL_METRIC_LABELS[name]} original')
    ax.set_ylabel(f'{PHYSICAL_METRIC_LABELS[name]} generated')
    ax.set_title(f'{name}  R^2={r2:.3f}')
fig.suptitle(f'{MODEL_NAME}: physical metrics parity')
save_figure(fig, f'{WORK_DIR}/physical_metrics_parity')
plt.show()
print('Physical metric R^2:', {k: round(v, 4) for k, v in phys_r2.items()})


#### Physical-metric agreement on the test split


In [ ]:
_phys_metrics_table_rows = []
phys_r2_table = {}
for _name in PHYSICAL_METRIC_NAMES:
    o, g = phys_orig[_name], phys_gen[_name]
    valid = np.isfinite(o) & np.isfinite(g)
    n_valid = int(valid.sum())
    o_v, g_v = o[valid], g[valid]
    if n_valid > 1:
        r2 = float(r2_score(o_v, g_v))
        pearson_r = float(np.corrcoef(o_v, g_v)[0, 1])
        mae = float(mean_absolute_error(o_v, g_v))
        rmse = float(np.sqrt(mean_squared_error(o_v, g_v)))
    else:
        r2 = pearson_r = mae = rmse = float('nan')
    if n_valid > 0:
        bias = float(g_v.mean() - o_v.mean())
        o_mean, o_std = float(o_v.mean()), float(o_v.std())
        g_mean, g_std = float(g_v.mean()), float(g_v.std())
    else:
        bias = o_mean = o_std = g_mean = g_std = float('nan')

    phys_r2_table[_name] = {
        'r2': r2, 'pearson_r': pearson_r, 'mae': mae, 'rmse': rmse, 'bias': bias,
        'orig_mean': o_mean, 'orig_std': o_std, 'gen_mean': g_mean, 'gen_std': g_std,
        'n': n_valid,
    }
    _phys_metrics_table_rows.append(
        (_name, r2, pearson_r, mae, rmse, bias, o_mean, o_std, g_mean, g_std, n_valid))

_header = (f'{"metric":22s} {"R2":>8s} {"pearson_r":>10s} {"MAE":>8s} {"RMSE":>8s} '
           f'{"bias":>8s} {"orig mean+-std":>18s} {"gen mean+-std":>18s} {"n":>6s}')
print(_header)
print('-' * len(_header))
for (_name, r2, pearson_r, mae, rmse, bias, o_mean, o_std, g_mean, g_std, n_valid) in _phys_metrics_table_rows:
    print(f'{_name:22s} {r2:8.4f} {pearson_r:10.4f} {mae:8.4f} {rmse:8.4f} '
          f'{bias:+8.4f} {o_mean:+7.3f}+-{o_std:<6.3f} {g_mean:+7.3f}+-{g_std:<6.3f} {n_valid:6d}')


#### s_z projections: original vs generated

These are the central-layer $s_z$ projections of the nanodot (z = floor(L/2)), cropped to the 39x39 physical grid; each column is annotated with that sample's physical-metric values.


In [ ]:
N_SHOW = 8
_idx_show = np.linspace(0, len(orig_39) - 1, N_SHOW).astype(int)
fig, axes = plt.subplots(3, N_SHOW, figsize=(2.1 * N_SHOW, 7.0))
for _col, _i in enumerate(_idx_show):
    _o, _g = orig_39[_i], gen_39[_i]
    _d = np.abs(_o - _g)
    for _row, (_img, _cmap, _vmin, _vmax) in enumerate(
            [(_o, 'coolwarm', -1.0, 1.0), (_g, 'coolwarm', -1.0, 1.0), (_d, 'magma', 0.0, 2.0)]):
        _ax = axes[_row, _col]
        _im = _ax.imshow(_img, cmap=_cmap, vmin=_vmin, vmax=_vmax, interpolation='nearest')
        _ax.set_xticks([]); _ax.set_yticks([])
    axes[0, _col].set_title(
        '\n'.join(f'{_n}={phys_orig[_n][_i]:+.3f}' for _n in PHYSICAL_METRIC_NAMES), fontsize=6)
    axes[2, _col].set_xlabel(
        '\n'.join(f'{_n}={phys_gen[_n][_i]:+.3f}' for _n in PHYSICAL_METRIC_NAMES), fontsize=6)
axes[0, 0].set_ylabel('original $s_z$', fontsize=9)
axes[1, 0].set_ylabel('generated $s_z$', fontsize=9)
axes[2, 0].set_ylabel('|difference|', fontsize=9)
fig.suptitle(f'{MODEL_NAME}: central-layer $s_z$ projections '
             f'(top titles = original metrics, bottom labels = generated metrics)')
fig.tight_layout()
save_figure(fig, f'{WORK_DIR}/sz_projections')
plt.show()


### SSIM and masked MSE

In [ ]:
ssim_vals = np.array([masked_ssim(o, g) for o, g in zip(orig_39, gen_39)])
mse_vals = np.array([masked_mse(o, g) for o, g in zip(orig_39, gen_39)])
print(f'SSIM       : mean={ssim_vals.mean():.4f}  std={ssim_vals.std():.4f}')
print(f'Masked MSE : mean={mse_vals.mean():.4f}  std={mse_vals.std():.4f}')


### Full cycle: theta -> DDPM -> encoder+head -> theta_hat

In [ ]:
@torch.no_grad()
def encoder_predict_theta(enc, x_norm39_np, batch_size=64):
    preds = []
    for i in range(0, len(x_norm39_np), batch_size):
        xb = torch.from_numpy(x_norm39_np[i:i + batch_size]).unsqueeze(1).float().to(DEVICE)
        y = enc(encoder_preprocess(xb)).cpu().numpy()
        preds.append(y)
    return np.concatenate(preds, axis=0)


theta_hat_scaled = encoder_predict_theta(encoder, gen_39)
theta_hat = INV_SCALER.inverse_transform(theta_hat_scaled)
theta_true = SPLIT['scaler'].inverse_transform(cond_te)

cycle_r2 = r2_score(theta_true, theta_hat, multioutput='raw_values')
cycle_mae = mean_absolute_error(theta_true, theta_hat, multioutput='raw_values')
cycle_rmse = np.sqrt(mean_squared_error(theta_true, theta_hat, multioutput='raw_values'))

print(f'{"param":8s} {"R2":>8s} {"MAE":>10s} {"RMSE":>10s}')
for i, name in enumerate(PARAM_NAMES):
    print(f'{name:8s} {cycle_r2[i]:8.4f} {cycle_mae[i]:10.4f} {cycle_rmse[i]:10.4f}')


### Per-magnetic-phase breakdown

In [ ]:
if labels is not None:
    idx_te_full = SPLIT['idx_te'][:n_test_eval]
    te_labels = labels[idx_te_full]
    phase_names = np.array([get_structure_label(c) for c in te_labels])
    rows = []
    for phase in sorted(set(phase_names.tolist())):
        sel = phase_names == phase
        if sel.sum() == 0:
            continue
        row = {'phase': phase, 'n': int(sel.sum()),
               'ssim': float(ssim_vals[sel].mean()),
               'masked_mse': float(mse_vals[sel].mean())}
        for name in PHYSICAL_METRIC_NAMES:
            o, g = phys_orig[name][sel], phys_gen[name][sel]
            valid = np.isfinite(o) & np.isfinite(g)
            row[f'r2_{name}'] = float(r2_score(o[valid], g[valid])) if valid.sum() > 1 else float('nan')
        rows.append(row)
    for row in rows:
        print(row)
else:
    print('No labels in the .npz -- per-phase breakdown skipped.')
    rows = []


### Save final metrics

In [ ]:
final_metrics = {
    'model_name': MODEL_NAME,
    'n_test_eval': int(n_test_eval),
    'physical_r2': {k: float(v) for k, v in phys_r2.items()},
    'ssim_mean': float(ssim_vals.mean()),
    'masked_mse_mean': float(mse_vals.mean()),
    'cycle_r2': {name: float(v) for name, v in zip(PARAM_NAMES, cycle_r2)},
    'cycle_mae': {name: float(v) for name, v in zip(PARAM_NAMES, cycle_mae)},
    'cycle_rmse': {name: float(v) for name, v in zip(PARAM_NAMES, cycle_rmse)},
    'per_phase': rows,
}
with open(METRICS_OUT, 'w') as f:
    json.dump(final_metrics, f, indent=2)
print(f'Saved final metrics to {METRICS_OUT}')
